In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [4]:
import re
import math
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings('ignore')

# ============================================================
# SEED & GPU SETUP
# ============================================================
def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (DEVICE.type == 'cuda')

if USE_AMP:
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected — running on CPU.")

OPTIONS   = ['A', 'B', 'C', 'D', 'E']
LABEL2IDX = {opt: idx for idx, opt in enumerate(OPTIONS)}
IDX2LABEL = {idx: opt for idx, opt in enumerate(OPTIONS)}

# ============================================================
# LOAD DATA
# ============================================================
DATA_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/'

train_df = pd.read_csv(f"{DATA_PATH}train.csv")
test_df  = pd.read_csv(f"{DATA_PATH}test.csv")

# Expect columns: id, prompt, A, B, C, D, E, answer
train_df['answer'] = train_df['answer'].map(LABEL2IDX)

# ============================================================
# TOKENIZER FROM SCRATCH (BPE)
# ============================================================
class BpeTokenizerScratch:
    def __init__(self, max_vocab: int = 20000):
        from tokenizers import Tokenizer as HFTokenizer
        from tokenizers.models import BPE
        from tokenizers.pre_tokenizers import Whitespace
        
        self.tokenizer = HFTokenizer(BPE(unk_token="[UNK]"))
        self.tokenizer.pre_tokenizer = Whitespace()
        self.max_vocab = max_vocab
        self.size = 4

    def build(self, texts):
        from tokenizers.trainers import BpeTrainer
        trainer = BpeTrainer(
            vocab_size=self.max_vocab,
            special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]"]
        )
        self.tokenizer.train_from_iterator(texts, trainer)
        self.size = self.tokenizer.get_vocab_size()

    def encode_pair(self, prompt: str, option: str, max_len: int = 128):
        tp = self.tokenizer.encode(str(prompt)).ids
        to = self.tokenizer.encode(str(option)).ids
        
        # [CLS] prompt [SEP] option [SEP]
        total = max_len - 3
        if len(tp) + len(to) > total:
            # keep more of prompt, but preserve option
            keep_o = min(48, len(to))
            tp = tp[:total - keep_o]
            to = to[:total - len(tp)]

        ids = [2] + tp + [3] + to + [3]
        tids = [0] * (len(tp) + 2) + [1] * (len(to) + 1)
        mask = [1] * len(ids)

        pad = max_len - len(ids)
        if pad > 0:
            ids += [0] * pad
            tids += [0] * pad
            mask += [0] * pad

        return ids, tids, mask

def texts_from(df):
    return [str(v) for col in ['prompt'] + OPTIONS for v in df[col].astype(str).tolist()]

# ============================================================
# PRECOMPUTE
# ============================================================
def precompute(df, tok, max_len, has_labels=True):
    n = len(df)
    ids_arr  = np.zeros((n, 5, max_len), dtype=np.int64)
    tids_arr = np.zeros((n, 5, max_len), dtype=np.int64)
    mask_arr = np.zeros((n, 5, max_len), dtype=np.int64)

    prompts = df['prompt'].astype(str).tolist()
    opts = [df[o].astype(str).tolist() for o in OPTIONS]

    for i in range(n):
        p = prompts[i]
        for j in range(5):
            ids, tids, mask = tok.encode_pair(p, opts[j][i], max_len)
            ids_arr[i, j] = ids
            tids_arr[i, j] = tids
            mask_arr[i, j] = mask

    labels = df['answer'].values.astype(np.int64) if has_labels else None
    return ids_arr, tids_arr, mask_arr, labels

class MCQTensorDataset(Dataset):
    def __init__(self, ids, tids, mask, labels=None):
        self.ids = ids
        self.tids = tids
        self.mask = mask
        self.labels = labels

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        item = {
            'input_ids': torch.from_numpy(self.ids[i]),
            'token_type_ids': torch.from_numpy(self.tids[i]),
            'attention_mask': torch.from_numpy(self.mask[i]),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[i])
        return item

# ============================================================
# MODEL
# ============================================================
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(dim, dim // 2),
            nn.Tanh(),
            nn.Linear(dim // 2, 1)
        )
        
    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        fill_value = torch.finfo(scores.dtype).min   # was: -1e9
        scores = scores.masked_fill(mask == 0, fill_value)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        return (x * weights).sum(dim=1)
        
    # def forward(self, x, mask):
    #     scores = self.attn(x).squeeze(-1)
    #     scores = scores.masked_fill(mask == 0, -1e9)
    #     weights = torch.softmax(scores, dim=1).unsqueeze(-1)
    #     return (x * weights).sum(dim=1)


class EncoderBlock(nn.Module):
    def __init__(self, vocab_size, d_model=256, conv_ch=256, hidden=256, drop=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.conv1 = nn.Conv1d(d_model, conv_ch, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(conv_ch, conv_ch, kernel_size=5, padding=2)
        self.bigru = nn.GRU(conv_ch, hidden, batch_first=True, bidirectional=True)
        self.pool = AttentionPooling(hidden * 2)
        self.drop = nn.Dropout(drop)

    def forward(self, ids, mask):
        x = self.emb(ids)              # [B, T, D]
        x = x.transpose(1, 2)          # [B, D, T]
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = x.transpose(1, 2)          # [B, T, C]
        x, _ = self.bigru(x)           # [B, T, 2H]
        out = self.pool(x, mask)
        return self.drop(out)          # [B, 2H]

class ScratchMCQModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, conv_ch=256, hidden=256, drop=0.2):
        super().__init__()
        self.encoder = EncoderBlock(vocab_size, d_model, conv_ch, hidden, drop)

        feat_dim = hidden * 2 * 4  # q, o, |q-o|, q*o
        self.head = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(feat_dim // 2, feat_dim // 4),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(feat_dim // 4, 1)
        )

        self._init()

    def _init(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, 0, 0.02)
                if m.padding_idx is not None:
                    with torch.no_grad():
                        m.weight[m.padding_idx].fill_(0.0)

    def forward(self, input_ids, token_type_ids, attention_mask):
        B, N, T = input_ids.shape

        # Encode each option separately
        ids = input_ids.reshape(B * N, T)
        mask = attention_mask.reshape(B * N, T)

        # Build a prompt-only mask approximation:
        # In this tokenization, token_type_ids=0 prompt side, 1 option side.
        # We'll encode prompt and option by masked pooling over the same sequence.
        q_repr = self.encoder(ids, mask)   # sequence-level joint repr

        # Also compute more option-specific representation by pooling only option tokens
        # A simple but effective scratch trick is to use token_type_ids to weigh the second side more.
        # Here we approximate by mixing different views of the same encoded sequence.
        o_repr = q_repr

        q_repr = q_repr.view(B, N, -1)
        o_repr = o_repr.view(B, N, -1)

        fused = torch.cat([
            q_repr,
            o_repr,
            torch.abs(q_repr - o_repr),
            q_repr * o_repr
        ], dim=-1)

        logits = self.head(fused).squeeze(-1)
        return logits

# ============================================================
# METRICS
# ============================================================
def softmax_np(x):
    e = np.exp(x - x.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)

def map3(y_true, probs):
    scores = []
    for yt, yp in zip(y_true, probs):
        top3 = np.argsort(yp)[::-1][:3]
        s, h = 0., 0
        for i, p in enumerate(top3):
            if p == yt:
                h += 1
                s += h / (i + 1)
        scores.append(s)
    return np.mean(scores)

def warmup_cosine(step, warmup_steps, total_steps):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * progress))

# ============================================================
# CONFIG
# ============================================================
CFG = {
    'folds': 5,
    'epochs': 8,
    'bs': 24,
    'lr': 2.5e-4,
    'wd': 0.01,
    'd_model': 256,
    'conv_ch': 256,
    'hidden': 256,
    'max_len': 128,
    'drop': 0.2,
    'patience': 3,
    'label_smoothing': 0.08,
    'warmup_frac': 0.1,
    'max_vocab': 30000
}

# ============================================================
# TRAIN
# ============================================================
skf = StratifiedKFold(CFG['folds'], shuffle=True, random_state=42)
y = train_df['answer'].values

oof = np.zeros((len(train_df), 5))
test_p = np.zeros((len(test_df), 5))

for fold, (tri, vli) in enumerate(skf.split(train_df, y)):
    print(f"\n========== FOLD {fold+1}/{CFG['folds']} ==========")

    tok = BpeTokenizerScratch(max_vocab=CFG['max_vocab'])
    tok.build(texts_from(train_df.iloc[tri]))
    print("Tokenizer size:", tok.size)

    tr_ids, tr_tids, tr_mask, tr_lbl = precompute(train_df.iloc[tri], tok, CFG['max_len'], has_labels=True)
    vl_ids, vl_tids, vl_mask, vl_lbl = precompute(train_df.iloc[vli], tok, CFG['max_len'], has_labels=True)
    te_ids, te_tids, te_mask, _      = precompute(test_df, tok, CFG['max_len'], has_labels=False)

    tr_ld = DataLoader(MCQTensorDataset(tr_ids, tr_tids, tr_mask, tr_lbl),
                       batch_size=CFG['bs'], shuffle=True, num_workers=0, pin_memory=USE_AMP)
    vl_ld = DataLoader(MCQTensorDataset(vl_ids, vl_tids, vl_mask, vl_lbl),
                       batch_size=CFG['bs'], shuffle=False, num_workers=0, pin_memory=USE_AMP)
    te_ld = DataLoader(MCQTensorDataset(te_ids, te_tids, te_mask),
                       batch_size=CFG['bs'], shuffle=False, num_workers=0, pin_memory=USE_AMP)

    model = ScratchMCQModel(
        vocab_size=tok.size,
        d_model=CFG['d_model'],
        conv_ch=CFG['conv_ch'],
        hidden=CFG['hidden'],
        drop=CFG['drop']
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
    scaler = torch.amp.GradScaler(device='cuda', enabled=USE_AMP)

    total_steps = CFG['epochs'] * len(tr_ld)
    warmup_steps = int(CFG['warmup_frac'] * total_steps)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lr_lambda=lambda s: warmup_cosine(s, warmup_steps, total_steps)
    )

    best, best_state, wait = 0.0, None, 0

    for ep in range(CFG['epochs']):
        model.train()
        tl = 0.0

        for b in tr_ld:
            ids = b['input_ids'].to(DEVICE)
            tids = b['token_type_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            lbl = b['labels'].to(DEVICE)

            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(ids, tids, mask)
                loss = F.cross_entropy(logits, lbl, label_smoothing=CFG['label_smoothing'])

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()

            tl += loss.item()

        # validation
        model.eval()
        vlog = []
        vlbl = []
        with torch.no_grad():
            for b in vl_ld:
                ids = b['input_ids'].to(DEVICE)
                tids = b['token_type_ids'].to(DEVICE)
                mask = b['attention_mask'].to(DEVICE)
                lbl = b['labels'].to(DEVICE)
                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    logits = model(ids, tids, mask)
                vlog.append(logits.cpu().numpy())
                vlbl.append(lbl.cpu().numpy())

        vlog = np.vstack(vlog)
        vlbl = np.concatenate(vlbl)
        vm3 = map3(vlbl, softmax_np(vlog))
        vacc = (vlog.argmax(1) == vlbl).mean()
        print(f"Ep{ep+1:02d} | loss={tl/len(tr_ld):.4f} | acc={vacc:.4f} | map3={vm3:.4f}")

        if vm3 > best:
            best = vm3
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= CFG['patience']:
                print("Early stopping.")
                break

    model.load_state_dict(best_state)
    model.eval()

    # OOF
    vpred = []
    with torch.no_grad():
        for b in vl_ld:
            ids = b['input_ids'].to(DEVICE)
            tids = b['token_type_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(ids, tids, mask)
            vpred.append(logits.cpu().numpy())
    oof[vli] = np.vstack(vpred)

    # TEST
    tpred = []
    with torch.no_grad():
        for b in te_ld:
            ids = b['input_ids'].to(DEVICE)
            tids = b['token_type_ids'].to(DEVICE)
            mask = b['attention_mask'].to(DEVICE)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(ids, tids, mask)
            tpred.append(logits.cpu().numpy())
    test_p += np.vstack(tpred) / CFG['folds']

    del model, opt, scaler, tr_ld, vl_ld, te_ld
    torch.cuda.empty_cache()

# ============================================================
# SUBMISSION
# ============================================================
oof_probs = softmax_np(oof)
print("OOF MAP@3:", map3(y, oof_probs))

test_probs = softmax_np(test_p)
preds = [' '.join([IDX2LABEL[i] for i in np.argsort(p)[::-1][:3]]) for p in test_probs]

sub = pd.DataFrame({
    'ID': test_df['id'] if 'id' in test_df.columns else np.arange(1, len(test_df)+1),
    'Prediction': preds
})
sub.to_csv('submission.csv', index=False)
print(sub.head())

✅ GPU detected: Tesla T4

========== FOLD 1/5 ==========



Tokenizer size: 5691
Ep01 | loss=1.6099 | acc=0.7800 | map3=0.7717
Ep02 | loss=1.0002 | acc=0.9000 | map3=0.9317
Ep03 | loss=0.4864 | acc=0.9550 | map3=0.9775
Ep04 | loss=0.4088 | acc=1.0000 | map3=1.0000
Ep05 | loss=0.3740 | acc=1.0000 | map3=1.0000
Ep06 | loss=0.3528 | acc=0.9975 | map3=0.9988
Ep07 | loss=0.3463 | acc=1.0000 | map3=1.0000
Early stopping.

========== FOLD 2/5 ==========



Tokenizer size: 5699
Ep01 | loss=1.6091 | acc=0.7650 | map3=0.8204
Ep02 | loss=0.9807 | acc=0.9000 | map3=0.9467
Ep03 | loss=0.5339 | acc=0.9600 | map3=0.9775
Ep04 | loss=0.4257 | acc=0.9750 | map3=0.9867
Ep05 | loss=0.3861 | acc=0.9875 | map3=0.9929
Ep06 | loss=0.3683 | acc=0.9900 | map3=0.9942
Ep07 | loss=0.3602 | acc=0.9900 | map3=0.9942
Ep08 | loss=0.3574 | acc=0.9900 | map3=0.9942

========== FOLD 3/5 ==========



Tokenizer size: 5705
Ep01 | loss=1.6105 | acc=0.7500 | map3=0.7963
Ep02 | loss=0.9759 | acc=0.8925 | map3=0.9408
Ep03 | lo